# Лабораторная работа №5

## Генерация тестовых данных

In [30]:
import numpy as np
import random

In [31]:
VEC_LEN = 4
NUM_CALLS = 100000

In [32]:
def generate_test_data(seed=740):
    random.seed(seed)
    np.random.seed(seed)
    
    vectors = []
    for i in range(NUM_CALLS + 1):
        vec = tuple(random.uniform(-1000, 1000) for _ in range(VEC_LEN))
        vectors.append(vec)
    
    pairs = [(vectors[i], vectors[i+1]) for i in range(NUM_CALLS)]
    return pairs

## Ctypes

In [33]:
import ctypes

lib = ctypes.CDLL('./libvector_dot.so')

class DotResult(ctypes.Structure):
    _fields_ = [("result", ctypes.c_double),
                ("error", ctypes.c_int)]

lib.dot_product.argtypes = [ctypes.POINTER(ctypes.c_double), ctypes.POINTER(ctypes.c_double)]
lib.dot_product.restype = DotResult

def dot_product_ctypes(a, b):
    a_array = (ctypes.c_double * VEC_LEN)(*a)
    b_array = (ctypes.c_double * VEC_LEN)(*b)
    result = lib.dot_product(a_array, b_array)
    if result.error:
        raise ValueError("NULL pointer in C function")
    return result.result

def run_test_ctypes():
    pairs = generate_test_data()
    results = []
    for a, b in pairs:
        results.append(dot_product_ctypes(a, b))
    return results

## CFFI

In [34]:
from cffi import FFI

ffi = FFI()

ffi.cdef("""
    #define VEC_LEN 4
    typedef struct {
        double result;
        int error;
    } DotResult;
    
    DotResult dot_product(const double *a, const double *b);
""")

lib = ffi.dlopen('./libvector_dot.so')

def dot_product_cffi(a, b):
    a_ptr = ffi.new("double[]", a)
    b_ptr = ffi.new("double[]", b)
    result = lib.dot_product(a_ptr, b_ptr)
    if result.error:
        raise ValueError("NULL pointer in C function")
    return result.result

def run_test_cffi():
    pairs = generate_test_data()
    results = []
    for a, b in pairs:
        results.append(dot_product_cffi(a, b))
    return results

## Cython

In [38]:
import pyximport
pyximport.install(
    setup_args={"include_dirs": [np.get_include()]},
    reload_support=True
)

from vector_dot_cython import dot_product_cython, run_test_cython

ImportError: Building module vector_dot_cython failed: ["ModuleNotFoundError: No module named 'distutils'\n"]

## Тестирование

In [ ]:
import time
import statistics

def benchmark(impl_name, run_func, num_runs=50):
    print(f"\nBenchmarking {impl_name}...")
    times = []
    
    for i in range(num_runs):
        start = time.perf_counter()
        run_func()
        end = time.perf_counter()
        elapsed = end - start
        times.append(elapsed)
        
        if (i + 1) % 10 == 0:
            print(f"  Completed {i + 1}/{num_runs} runs")
    
    mean = statistics.mean(times)
    median = statistics.median(times)
    std_dev = statistics.stdev(times)
    min_time = min(times)
    max_time = max(times)
    
    return {
        'min': min_time,
        'max': max_time,
        'mean': mean,
        'median': median,
        'std_dev': std_dev
    }

In [36]:
benchmark("cffi", run_test_cffi)


Benchmarking cffi...
  Completed 10/50 runs
  Completed 20/50 runs
  Completed 30/50 runs
  Completed 40/50 runs
  Completed 50/50 runs


{'min': 0.10140524999951595,
 'max': 0.14130670800022926,
 'mean': 0.11020015575999423,
 'median': 0.1096831459999521,
 'std_dev': 0.006191776242335705}